# LP-FT RAD-DINO — 13 nhãn CheXpert (U-Ones, ASL, cache 256px)

**Luồng:** so frontal vs multi-view (frozen) → cache 256px (Drive+local, resumable) → **LP** (head trên frozen CLS) → **FT** (last-3 block + head, ASL) → eval test.
- 13 nhãn (bỏ Support Devices), U-Ones (-1→1, blank→0), split **70/10/20 theo subject**.
- Loss **Asymmetric Loss**; early-stop **macro-AUPRC** (ưu tiên precision/recall).
- **CHỈ ĐỌC** `manifest_cohort.parquet` + `embeddings_raddino_ft_cls.npy`; mọi output ghi file/thư mục MỚI (không đè dataset gốc).
> Chỉnh `SRC_DIR`/`WRITE_DIR` cho khớp máy bạn. Cache + checkpoint ghi vào Drive RIÊNG của bạn (không ghi vào thư mục chia sẻ).

## 1. Env + config

In [ ]:
import os, io, time, glob, math, shutil, numpy as np, pandas as pd, torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
try: drive.mount('/content/drive')
except Exception as e: print('drive:', e)
# ADC đã có sẵn (đã auth BigQuery/GCS trước đó). Nếu chưa: from google.colab import auth; auth.authenticate_user()

# ---- ĐƯỜNG DẪN (chỉnh cho khớp) ----
SRC_DIR   = '/content/drive/MyDrive/MIMIC_CXR'          # ĐỌC: manifest_cohort.parquet + embeddings_raddino_*
WRITE_DIR = '/content/drive/MyDrive/lpft_out'           # GHI: checkpoint, metrics (Drive RIÊNG của bạn)
CACHE_DIR = '/content/drive/MyDrive/lpft_cache256'      # cache uint8 256 (Drive riêng, resumable)
LOCAL_CACHE = '/content/lpft_cache256'                  # copy về local để train nhanh
for d in (WRITE_DIR, CACHE_DIR, LOCAL_CACHE): os.makedirs(d, exist_ok=True)

BILLING_PROJECT='mimic-ehr-v310'; REPO='microsoft/rad-dino'; BUCKET='mimic-cxr-jpg-2.1.0.physionet.org'
THREADS=32; CACHE_RES=256; TRAIN_RES=224; CACHE_SHARD=4000; N_UNFREEZE=3
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

proc=AutoImageProcessor.from_pretrained(REPO); MEAN, STD = proc.image_mean, proc.image_std
print('mean/std:', MEAN, STD)

CHEXPERT_LABELS=['Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax','Support Devices']
LABELS=[c for c in CHEXPERT_LABELS if c!='Support Devices']   # 13 nhãn
print('LABELS:', len(LABELS))

# helpers dùng chung
def resolve_uones(L): return np.where(np.isnan(L),0.0,np.where(L==-1,1.0,L)).astype('float32')
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, fbeta_score
GRID=np.arange(0.05,0.96,0.01)
def bthr(y,p):
    s=[fbeta_score(y,(p>=t).astype(int),beta=2,zero_division=0) for t in GRID]; return float(GRID[int(np.argmax(s))])
def macro_ap_from_P(P, Lraw):
    aps=[]
    for j in range(len(LABELS)):
        col=Lraw[:,j]; keep=col!=-1; yt=np.where(np.isnan(col[keep]),0.,col[keep]).astype(int)
        if yt.sum()>0: aps.append(average_precision_score(yt, P[keep,j]))
    return float(np.mean(aps))

## 2. cm + split 70/10/20 theo subject

In [ ]:
cm = pd.read_parquet(os.path.join(SRC_DIR,'manifest_cohort.parquet'))
cm['rel'] = cm['jpg_path'].str.split(BUCKET+'/').str[-1]
print('cm:', cm.shape, '| views:', cm.view.value_counts().to_dict())

rng=np.random.default_rng(42); uniq=np.unique(cm.subject_id.values); rng.shuffle(uniq)
n=len(uniq); s1,s2=int(.70*n),int(.80*n)
tr_s,va_s,te_s=set(uniq[:s1]),set(uniq[s1:s2]),set(uniq[s2:])
cm['mysplit']=np.where(cm.subject_id.isin(tr_s),'train',
               np.where(cm.subject_id.isin(va_s),'validate','test'))
assert not (tr_s&te_s) and not (tr_s&va_s) and not (va_s&te_s)
print('subject 70/10/20 (ảnh):', cm.mysplit.value_counts().to_dict())
print('frontal:', int((cm.view=='frontal').sum()))

## 2b. So sánh frontal-only vs multi-view (frozen probe) — CHẠY TRƯỚC để chốt view
Cùng 13 nhãn U-Ones, cùng split subject, **cùng chấm trên FRONTAL test**. Chọn config macro-AUPRC cao hơn.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
emb=np.load(os.path.join(SRC_DIR,'embeddings_raddino_ft_cls.npy'))
eidx=pd.read_parquet(os.path.join(SRC_DIR,'embeddings_raddino_index.parquet'))
mm=eidx[['dicom_id']].merge(cm[['dicom_id','view','mysplit']+LABELS], on='dicom_id', how='left')
Lp=mm[LABELS].to_numpy('float32'); sp=mm['mysplit'].values; fr=(mm.view=='frontal').values
tr=sp=='train'; va=sp=='validate'; te=sp=='test'

def probe(train_mask, tag):
    sc=StandardScaler().fit(emb[train_mask]); Xa=sc.transform(emb).astype('float32'); out=[]
    for j,c in enumerate(LABELS):
        col=Lp[:,j]; y=resolve_uones(col)
        clf=LogisticRegression(max_iter=2000,class_weight='balanced',n_jobs=-1).fit(Xa[train_mask], y[train_mask])
        vam=va&fr&(col!=-1); tem=te&fr&(col!=-1)     # eval trên FRONTAL, loại -1
        pv=clf.predict_proba(Xa[vam])[:,1]; yv=np.where(np.isnan(col[vam]),0.,col[vam]).astype(int)
        pt=clf.predict_proba(Xa[tem])[:,1]; yt=np.where(np.isnan(col[tem]),0.,col[tem]).astype(int)
        thr=bthr(yv,pv); pred=(pt>=thr).astype(int)
        out.append(dict(config=tag,label=c,
            auroc=roc_auc_score(yt,pt) if 0<yt.sum()<len(yt) else np.nan,
            auprc=average_precision_score(yt,pt) if yt.sum()>0 else np.nan,
            precision=precision_score(yt,pred,zero_division=0),
            recall=recall_score(yt,pred,zero_division=0),
            f2=fbeta_score(yt,pred,beta=2,zero_division=0)))
    return out

resv=pd.DataFrame(probe(tr&fr,'frontal') + probe(tr,'multiview'))
resv.to_csv(os.path.join(WRITE_DIR,'viewcompare_frozen.csv'), index=False)
print('== macro (eval trên frontal test) ==')
print(resv.groupby('config')[['auroc','auprc','precision','recall','f2']].mean().round(4))
print('\n-> chọn VIEWS cho các bước sau (mặc định frontal):')

## 3. Cache 256px (Drive resumable → copy local)

In [ ]:
VIEWS='frontal'    # 'frontal' hoặc 'all' — chốt theo kết quả 2b
sel = cm[cm.view=='frontal'].reset_index(drop=True) if VIEWS=='frontal' else cm.reset_index(drop=True)
print('cache số ảnh:', len(sel))

from google.cloud import storage
import google.auth
from google.auth.transport.requests import AuthorizedSession
from requests.adapters import HTTPAdapter
from concurrent.futures import ThreadPoolExecutor

tf256=T.Compose([T.Resize(CACHE_RES), T.CenterCrop(CACHE_RES)])
_bucket=None
def _get_bucket():
    global _bucket
    if _bucket is None:
        creds,_=google.auth.default(); sess=AuthorizedSession(creds)
        sess.mount('https://', HTTPAdapter(pool_connections=THREADS,pool_maxsize=THREADS))
        _bucket=storage.Client(project=BILLING_PROJECT,credentials=creds,_http=sess).bucket(BUCKET,user_project=BILLING_PROJECT)
    return _bucket
def _one256(rel):
    try:
        im=Image.open(io.BytesIO(_get_bucket().blob(rel).download_as_bytes())).convert('RGB')
        return np.asarray(tf256(im), dtype=np.uint8)     # (256,256,3)
    except Exception: return None
class BatchDS(Dataset):
    def __init__(s,df,bs): s.rel=df.rel.tolist(); s.id=df.dicom_id.tolist(); s.bs=bs
    def __len__(s): return (len(s.id)+s.bs-1)//s.bs
    def __getitem__(s,b):
        lo=b*s.bs; hi=min(lo+s.bs,len(s.id)); ids=s.id[lo:hi]; rels=s.rel[lo:hi]
        with ThreadPoolExecutor(THREADS) as ex: ims=list(ex.map(_one256,rels))
        keep=[(i,im) for i,im in zip(ids,ims) if im is not None]
        if not keep: return [],None
        return [k[0] for k in keep], np.stack([k[1] for k in keep])

# resume: đọc mọi cacheidx_*.parquet đã có
done=set()
for f in glob.glob(os.path.join(CACHE_DIR,'cacheidx_*.parquet')): done|=set(pd.read_parquet(f).dicom_id)
todo=sel[~sel.dicom_id.isin(done)].reset_index(drop=True)
print('đã cache:', len(done), '| còn:', len(todo))

sh=len(glob.glob(os.path.join(CACHE_DIR,'cache_*.npy')))
buf_id=[]; buf_im=[]
def flush():
    global sh,buf_id,buf_im
    np.save(os.path.join(CACHE_DIR,f'cache_{sh:05d}.npy'), np.concatenate(buf_im,0))
    pd.DataFrame([{'dicom_id':d,'shard':sh,'row':k} for k,d in enumerate(buf_id)]
                 ).to_parquet(os.path.join(CACHE_DIR,f'cacheidx_{sh:05d}.parquet'), index=False)
    sh+=1; buf_id=[]; buf_im=[]
if len(todo):
    loader=DataLoader(BatchDS(todo,256), batch_size=None, num_workers=8, prefetch_factor=4)
    seen=0; t0=time.time(); pbar=tqdm(loader,total=len(loader))
    for ids,ims in pbar:
        if ims is None: continue
        buf_id+=list(ids); buf_im.append(ims); seen+=len(ids)
        pbar.set_postfix(img_s=round(seen/(time.time()-t0),1))
        if sum(a.shape[0] for a in buf_im)>=CACHE_SHARD: flush()
    if buf_id: flush()

# gộp index + copy local
cidx=pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(CACHE_DIR,'cacheidx_*.parquet')))], ignore_index=True)
cidx.to_parquet(os.path.join(CACHE_DIR,'cache_index.parquet'), index=False)
for f in sorted(glob.glob(os.path.join(CACHE_DIR,'cache_*.npy'))):
    dst=os.path.join(LOCAL_CACHE, os.path.basename(f))
    if not os.path.exists(dst): shutil.copy2(f,dst)
shutil.copy2(os.path.join(CACHE_DIR,'cache_index.parquet'), os.path.join(LOCAL_CACHE,'cache_index.parquet'))
print('cache xong:', len(cidx), 'ảnh |', len(glob.glob(os.path.join(LOCAL_CACHE,'cache_*.npy'))), 'shard local')

In [ ]:
# ---- nạp cache local để train (memmap) ----
cidx=pd.read_parquet(os.path.join(LOCAL_CACHE,'cache_index.parquet'))
shard_arrs={int(s): np.load(os.path.join(LOCAL_CACHE,f'cache_{int(s):05d}.npy'), mmap_mode='r') for s in cidx.shard.unique()}
id2loc={d:(int(s),int(r)) for d,s,r in zip(cidx.dicom_id,cidx.shard,cidx.row)}
print('cache index:', len(id2loc), 'ảnh |', len(shard_arrs), 'shard')

# selc = tập ảnh sẽ dùng (theo VIEWS) có trong cache + split
selc = sel[sel.dicom_id.isin(id2loc)].reset_index(drop=True)
trm=(selc.mysplit=='train').values; vam=(selc.mysplit=='validate').values; tem=(selc.mysplit=='test').values
print('selc train/val/test:', int(trm.sum()), int(vam.sum()), int(tem.sum()))

## 4. Pha 1 — LP (head trên frozen CLS thô, KHÔNG scale)

In [ ]:
class ASL(nn.Module):
    def __init__(s,gn=4,gp=1,clip=.05,eps=1e-8):
        super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xs=torch.sigmoid(logits); xp=xs; xn=1-xs
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        l=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y)
        return -(l*(1-pt)**g).mean()

# frozen CLS thô cho selc (KHÔNG StandardScaler -> head chuyển sang FT đúng)
emb=np.load(os.path.join(SRC_DIR,'embeddings_raddino_ft_cls.npy'))
eidx=pd.read_parquet(os.path.join(SRC_DIR,'embeddings_raddino_index.parquet'))
erow={d:i for i,d in enumerate(eidx.dicom_id)}
E=emb[selc.dicom_id.map(erow).to_numpy()]                # (Nsel,768)
Lraw_c=selc[LABELS].to_numpy('float32')
Ylp=resolve_uones(Lraw_c)
Et=torch.tensor(E,dtype=torch.float32); Yt=torch.tensor(Ylp,dtype=torch.float32)

head=nn.Linear(768,len(LABELS)).to(device)
opt=torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.05); crit=ASL()
tr_i=np.where(trm)[0]; va_i=np.where(vam)[0]
best=-1; best_state=None; bad=0; PATIENCE=5
for ep in range(30):
    head.train(); perm=np.random.default_rng(ep).permutation(tr_i)
    for i in range(0,len(perm),1024):
        b=perm[i:i+1024]; x=Et[b].to(device); y=Yt[b].to(device)
        opt.zero_grad(); loss=crit(head(x),y); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad(): Pv=torch.sigmoid(head(Et[va_i].to(device))).cpu().numpy()
    ap=macro_ap_from_P(Pv, Lraw_c[va_i])
    print(f'LP ep{ep:02d} val macro-AUPRC={ap:.4f}')
    if ap>best: best=ap; best_state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}; bad=0
    else:
        bad+=1
        if bad>=PATIENCE: print('early stop LP'); break
head.load_state_dict(best_state); torch.save(head.state_dict(), os.path.join(WRITE_DIR,'lpft_head_lp.pt'))
print('LP best val macro-AUPRC:', round(best,4))

## 5. Pha 2 — FT (last-3 block + layernorm + head), ASL, early-stop macro-AUPRC

In [ ]:
model=AutoModel.from_pretrained(REPO).to(device)
# verify tên module (RAD-DINO = Dinov2Model)
assert hasattr(model,'encoder') and hasattr(model.encoder,'layer'), [n for n,_ in model.named_children()]
assert hasattr(model,'layernorm'), [n for n,_ in model.named_children()]
for p in model.parameters(): p.requires_grad=False
for blk in model.encoder.layer[-N_UNFREEZE:]:
    for p in blk.parameters(): p.requires_grad=True
for p in model.layernorm.parameters(): p.requires_grad=True
head=nn.Linear(768,len(LABELS)).to(device); head.load_state_dict(torch.load(os.path.join(WRITE_DIR,'lpft_head_lp.pt')))
print('trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad)+sum(p.numel() for p in head.parameters()))

NORM=T.Normalize(MEAN,STD)
aug_tr=T.Compose([T.RandomResizedCrop(TRAIN_RES,scale=(0.8,1.0),ratio=(0.9,1.1)), T.ToTensor(), NORM])  # KHÔNG hflip
aug_ev=T.Compose([T.Resize(TRAIN_RES), T.CenterCrop(TRAIN_RES), T.ToTensor(), NORM])
class CacheDS(Dataset):
    def __init__(s,df,train):
        s.ids=df.dicom_id.tolist(); s.y=resolve_uones(df[LABELS].to_numpy('float32')); s.tf=aug_tr if train else aug_ev
    def __len__(s): return len(s.ids)
    def __getitem__(s,i):
        sd,rw=id2loc[s.ids[i]]; arr=np.array(shard_arrs[sd][rw])
        return s.tf(Image.fromarray(arr)), torch.tensor(s.y[i])

tr_df=selc[trm].reset_index(drop=True); va_df=selc[vam].reset_index(drop=True); te_df=selc[tem].reset_index(drop=True)
trl=DataLoader(CacheDS(tr_df,True), batch_size=64, shuffle=True,  num_workers=8, pin_memory=True, drop_last=True, persistent_workers=True)
vll=DataLoader(CacheDS(va_df,False),batch_size=128,shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

opt=torch.optim.AdamW([{'params':[p for p in model.parameters() if p.requires_grad],'lr':1e-5},
                       {'params':head.parameters(),'lr':1e-4}], weight_decay=0.05)
EPOCHS_FT=8; sched=torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-5,1e-4], total_steps=len(trl)*EPOCHS_FT, pct_start=0.1)
crit=ASL()
def fwd(x):
    h=model(pixel_values=x, interpolate_pos_encoding=True).last_hidden_state
    return head(h[:,0])
def eval_P(loader, df):
    model.eval(); head.eval(); ps=[]
    with torch.no_grad(), torch.autocast('cuda',dtype=torch.bfloat16):
        for x,y in loader: ps.append(torch.sigmoid(fwd(x.to(device))).float().cpu().numpy())
    P=np.concatenate(ps,0); return P, macro_ap_from_P(P, df[LABELS].to_numpy('float32'))

best=-1; bad=0; PATIENCE=3
for ep in range(EPOCHS_FT):
    model.train(); head.train(); t0=time.time()
    for x,y in tqdm(trl, desc=f'FT ep{ep}'):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        opt.zero_grad()
        with torch.autocast('cuda',dtype=torch.bfloat16): loss=crit(fwd(x),y)
        loss.backward(); opt.step(); sched.step()
    _,ap=eval_P(vll, va_df)
    print(f'FT ep{ep} val macro-AUPRC={ap:.4f} ({time.time()-t0:.0f}s)')
    if ap>best:
        best=ap; bad=0
        torch.save({'backbone':model.state_dict(),'head':head.state_dict(),'val_auprc':ap,
                    'labels':LABELS,'N_UNFREEZE':N_UNFREEZE,'VIEWS':VIEWS},
                   os.path.join(WRITE_DIR,'raddino_lpft_uones_best.pt'))
        print('  ✓ saved checkpoint')
    else:
        bad+=1
        if bad>=PATIENCE: print('early stop FT'); break
print('FT best val macro-AUPRC:', round(best,4))

## 6. Eval test — per-label AUROC/AUPRC/precision/recall/F2 (LP-FT vs LP-only)

In [ ]:
ck=torch.load(os.path.join(WRITE_DIR,'raddino_lpft_uones_best.pt'))
model.load_state_dict(ck['backbone']); head.load_state_dict(ck['head'])
tel=DataLoader(CacheDS(te_df,False), batch_size=128, num_workers=8, pin_memory=True)
Pte,_=eval_P(tel, te_df); Pva,_=eval_P(vll, va_df)
Lr_te=te_df[LABELS].to_numpy('float32'); Lr_va=va_df[LABELS].to_numpy('float32')
rows=[]
for j,c in enumerate(LABELS):
    cv=Lr_va[:,j]; kv=cv!=-1; yv=np.where(np.isnan(cv[kv]),0.,cv[kv]).astype(int); pv=Pva[kv,j]
    ct=Lr_te[:,j]; kt=ct!=-1; yt=np.where(np.isnan(ct[kt]),0.,ct[kt]).astype(int); pt=Pte[kt,j]
    thr=bthr(yv,pv); pred=(pt>=thr).astype(int)
    rows.append(dict(label=c,
        auroc=roc_auc_score(yt,pt) if 0<yt.sum()<len(yt) else np.nan,
        auprc=average_precision_score(yt,pt) if yt.sum()>0 else np.nan,
        precision=precision_score(yt,pred,zero_division=0),
        recall=recall_score(yt,pred,zero_division=0),
        f2=fbeta_score(yt,pred,beta=2,zero_division=0), thr=round(thr,3),
        prevalence=round(float(yt.mean()),4), n_pos=int(yt.sum())))
mt=pd.DataFrame(rows); mt.to_csv(os.path.join(WRITE_DIR,'lpft_metrics.csv'), index=False)
print(mt.to_string(index=False))
print('\nMACRO LP-FT:', mt[['auroc','auprc','precision','recall','f2']].mean().round(4).to_dict())
print('(so với LP-only frozen A/B U-Ones macro-AUROC ~0.787 / AUPRC ~0.37)')